# Checkpoint Utility

源码导航：[`unwrap_model`](../../../core/utils/walkie_checkpoint.py#L32)、[`save_walkie_checkpoint`](../../../core/utils/walkie_checkpoint.py#L81)、[`load_walkie_checkpoint`](../../../core/utils/walkie_checkpoint.py#L139)、[`apply_walkie_checkpoint`](../../../core/utils/walkie_checkpoint.py#L174)、[`resolve_resume_path`](../../../core/utils/walkie_checkpoint.py#L205)。

训练断点不只是 `model.state_dict()`。为了让恢复后的训练轨迹尽量连续，需要保存：

$$
\mathcal{C}=\{\theta,\;s_{opt},\;s_{scaler},\;s_{schedule},\;t,\;stage,\;rng\}
$$

其中 $\theta$ 是模型参数，$s_{opt}$ 是优化器状态，$t$ 是当前 step，`rng` 覆盖 Python/NumPy/torch/CUDA 随机数状态。这个模块的改进点是把 DDP/torch.compile 包装剥离、版本校验、架构校验、latest/best/step 解析集中到一处，训练脚本只需要调用统一接口。

In [ ]:
from __future__ import annotations

import sys
import tempfile
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.model.walkie import WalkieConfig, WalkieForCausalLM
from core.utils.walkie_checkpoint import (
    apply_walkie_checkpoint,
    load_walkie_checkpoint,
    resolve_resume_path,
    save_walkie_checkpoint,
    unwrap_model,
)

## 1. 保存与恢复 tiny 模型

In [ ]:
cfg = WalkieConfig(
    vocab_size=64, block_size=32, n_embd=32, n_layer=1,
    n_head=4, n_head_kv=2, head_dim=8, d_ffn=64,
)
model = WalkieForCausalLM(cfg)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

with tempfile.TemporaryDirectory() as tmp:
    out_dir = Path(tmp)
    path = save_walkie_checkpoint(
        out_dir,
        model=model,
        optimizers={'adamw': optimizer},
        scaler=None,
        schedule_state={'step': 3},
        step=3,
        stage='main',
        best_metric=1.23,
        model_cfg=cfg.to_dict(),
        train_cfg={'batch_size': 2},
        format='latest',
    )
    print('saved:', path.name)
    print('resolved:', resolve_resume_path(out_dir).name)

    payload = load_walkie_checkpoint(path, expected_model_cfg=cfg.to_dict())
    restored = WalkieForCausalLM(cfg)
    info = apply_walkie_checkpoint(payload, model=restored, optimizers=None, scaler=None)
    print(info)

## 2. 包装剥离

In [ ]:
class FakeCompileWrapper(torch.nn.Module):
    def __init__(self, inner):
        super().__init__()
        self._orig_mod = inner

wrapped = FakeCompileWrapper(model)
print(unwrap_model(wrapped) is model)

---

## 延伸阅读与参考资料

### PyTorch 官方文档
- **Saving and Loading Models**: [tutorial](https://pytorch.org/tutorials/beginner/saving_loading_models.html)
- **Automatic Mixed Precision examples**: [docs](https://pytorch.org/docs/stable/notes/amp_examples.html)
- **Distributed Checkpoint**: [docs](https://pytorch.org/docs/stable/distributed.checkpoint.html)

### 工程实践
- **torch.save / torch.load serialization notes**: [docs](https://pytorch.org/docs/stable/notes/serialization.html)
- **Fully Sharded Data Parallel checkpointing patterns**: [docs](https://pytorch.org/docs/stable/fsdp.html)